# 05 - VideoMAE on person crops (Colab A100 / L4)

Same setup as `04_colab_train.ipynb` (code, configs and `secrets/kaggle.json` on Drive `MyDrive/DataSlayer26/`).
`configs/colab_videomae.yaml`: VideoMAE ViT-B (Kinetics-400) over 16 frames of 224x224 person crops. The crops come from
the YOLO11s-pose cache: copied from Drive `cache/pose_cache.tgz` if the laptop uploaded it, else extracted here with
`cli.extract` (~17 min on L4). Every test clip is predicted on its own (competition FAQ 27).

Plan: folds 0-2 first (baseline ConvNeXt-T on the same folds: clip F1 0.615, AUC 0.649; pose head fold 0: 0.641), then
folds 3-4 into the same run directory.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/DataSlayer26"
for need in ["src/training/engine.py", "configs/colab_videomae.yaml", "secrets/kaggle.json"]:
    if not os.path.exists(f"{ROOT}/{need}"):
        raise FileNotFoundError(f"{ROOT}/{need} missing: wait for Drive desktop sync on the laptop")
%cd {ROOT}
!pip -q install timm accelerate kaggle
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
os.environ["KAGGLE_CONFIG_DIR"] = f"{ROOT}/secrets"
os.environ["PYTHONPATH"] = f"{ROOT}/src"
if not os.path.exists("/content/data/sample_submission.csv"):
    # A cancelled download leaves a partial zip that the Kaggle CLI would skip as "up to date".
    !rm -f /content/data-slayer-4.zip
    !rm -rf /content/data
    !kaggle competitions download -c data-slayer-4 -p /content
    !unzip -q -o /content/data-slayer-4.zip -d /content/data
!ls /content/data

In [ ]:
import torch

CONFIG = "configs/colab_videomae.yaml"  # VideoMAE on person crops
SET = "train.folds=[0,1,2]"  # second session: "train.folds=[3,4]" (same run dir, CV pools all folds)
# 16 clips/step; below 40 GB (L4 24 GB) keep the same effective step as 8 x 2.
if torch.cuda.get_device_properties(0).total_memory < 35 * 2**30:
    SET += " train.batch_size=8 train.grad_accum=2"
print(torch.cuda.get_device_name(0), "| SET:", SET)
if not os.path.exists("/content/cache/videos.csv"):
    !python -m cli.prepare --config {CONFIG} --set {SET} --workers 8

In [ ]:
POSE_TAR = f"{ROOT}/cache/pose_cache.tgz"
if not os.path.exists("/content/cache/features/yolo11s-pose_360x640_s2/test"):
    if os.path.exists(POSE_TAR):
        !mkdir -p /content/cache/features && tar -xzf {POSE_TAR} -C /content/cache/features
    else:
        !pip -q install ultralytics
        !python -m cli.extract --config configs/feat_pose.yaml --set paths.data_dir=/content/data paths.cache_dir=/content/cache --workers 8
!ls /content/cache/features/yolo11s-pose_360x640_s2/train | wc -l

In [ ]:
!python -m cli.train --config {CONFIG} --set {SET}

In [ ]:
!python -m cli.predict --config {CONFIG} --set {SET}
!ls -la {ROOT}/outputs/videomae_b_crop {ROOT}/outputs/videomae_b_crop/fold*